# CITD Image Processing — Inference and Review UI

This notebook loads an already-trained `best.pt` model and runs image/video inference. It does not download the Roboflow dataset or retrain the detector. Hosted dependency setup is delegated to `scripts/bootstrap-kaggle.py`; add a Kaggle Dataset containing `best.pt`, then use the widgets below to review annotated video and OCR output.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.getenv("LPR_REPO_URL", "https://github.com/cuongmn2011/CITD_ImageProcessing.git")
REPO_REF = os.getenv("LPR_REPO_REF", "develop")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
REPO_DIR = WORK_ROOT / "CITD_ImageProcessing"

def run(command, *, cwd=REPO_DIR, env=None):
    print("$", " ".join(str(part) for part in command))
    completed = subprocess.run(command, cwd=cwd, env=env, check=False, text=True)
    if completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command)
    return completed

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)], cwd=WORK_ROOT)
else:
    run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR)
    run(["git", "checkout", REPO_REF], cwd=REPO_DIR)
    run(["git", "reset", "--hard", f"origin/{REPO_REF}"], cwd=REPO_DIR)

print("Repository:", REPO_DIR)
print("Revision:", REPO_REF)

In [ ]:
# Hosted runtimes use their existing PyTorch. Local runs use the locked uv env.
import importlib.util

USE_SYSTEM_ENV = Path("/kaggle").is_dir() or Path("/content").is_dir()

if USE_SYSTEM_ENV:
    run([sys.executable, "scripts/bootstrap-kaggle.py", "--mode", "inference"], cwd=REPO_DIR)
    LPR_COMMAND = [sys.executable, "-m", "lpr.cli"]
else:
    UV = shutil.which("uv")
    if UV is None:
        run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                "--no-input",
                "-q",
                "uv",
            ],
            cwd=REPO_DIR,
        )
        UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv executable was not found after installation")
    run([UV, "sync", "--extra", "vision", "--extra", "ocr"])
    LPR_COMMAND = [UV, "run", "lpr"]

ENV = os.environ.copy()
ENV["MPLBACKEND"] = "Agg"
ENV["PYTHONPATH"] = os.pathsep.join(
    filter(None, [str(REPO_DIR / "src"), ENV.get("PYTHONPATH", "")])
)


In [ ]:
# Install the system OCR binary when running on a fresh Kaggle/Colab runtime.
if shutil.which("tesseract") is None:
    run(["apt-get", "update", "-qq"], cwd=REPO_DIR)
    run(["apt-get", "install", "-y", "-qq", "tesseract-ocr"], cwd=REPO_DIR)
print("Tesseract:", shutil.which("tesseract"))

In [ ]:
# Resolve a model uploaded as a Kaggle Dataset, a training zip, or a local models/best.pt.
import zipfile

configured_model = os.getenv("LPR_MODEL_PATH")
model_candidates = []
if configured_model:
    model_candidates.append(Path(configured_model).expanduser())
if Path("/kaggle/input").is_dir():
    model_candidates.extend(sorted(Path("/kaggle/input").glob("**/best.pt")))
model_candidates.append(REPO_DIR / "models/best.pt")
model_candidates = [path for path in model_candidates if path.is_file()]

if not model_candidates and Path("/kaggle/input").is_dir():
    archive_candidates = sorted(Path("/kaggle/input").glob("**/*.zip"))
    for archive_path in archive_candidates:
        with zipfile.ZipFile(archive_path) as archive:
            best_members = [
                member
                for member in archive.namelist()
                if Path(member).name == "best.pt"
            ]
            if best_members:
                extracted_model = WORK_ROOT / "model_artifact" / "best.pt"
                extracted_model.parent.mkdir(parents=True, exist_ok=True)
                extracted_model.write_bytes(archive.read(best_members[0]))
                model_candidates.append(extracted_model)
                break

if not model_candidates:
    raise FileNotFoundError(
        "Add best.pt or a training zip as a Kaggle Dataset, or set LPR_MODEL_PATH"
    )
MODEL_PATH = model_candidates[0].resolve()
print("Model:", MODEL_PATH)
print("Model size:", f"{MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MB")

VIDEO_CANDIDATES = sorted(Path("/kaggle/input").glob("**/*.mp4")) if Path("/kaggle/input").is_dir() else []
DEFAULT_VIDEO = str(VIDEO_CANDIDATES[0]) if VIDEO_CANDIDATES else os.getenv("LPR_VIDEO_INPUT", "")
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else WORK_ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Default video:", DEFAULT_VIDEO or "not set")

In [ ]:
# Interactive video inference and inline playback.
import ipywidgets as widgets
from IPython.display import FileLink, Video, clear_output, display

video_input = widgets.Text(value=DEFAULT_VIDEO, description="Video:", layout=widgets.Layout(width="95%"))
video_output = widgets.Text(value=str(OUTPUT_ROOT / "annotated.mp4"), description="Output:", layout=widgets.Layout(width="95%"))
OCR_OPTIONS = ("tesseract",)
if importlib.util.find_spec("easyocr") is not None:
    OCR_OPTIONS += ("easyocr",)
else:
    print("EasyOCR is unavailable; the demo will use Tesseract.")
ocr_backend = widgets.Dropdown(options=OCR_OPTIONS, value="tesseract", description="OCR:")
device_input = widgets.Text(value="auto", description="Device:")
max_frames = widgets.IntText(value=0, min=0, description="Max frames:")
run_button = widgets.Button(description="Run video inference", button_style="primary", icon="play")
video_output_area = widgets.Output()

def run_video_inference(_button):
    with video_output_area:
        clear_output(wait=True)
        input_path = Path(video_input.value).expanduser()
        output_path = Path(video_output.value).expanduser()
        if not input_path.is_file():
            print(f"Video file not found: {input_path}")
            return
        output_path.parent.mkdir(parents=True, exist_ok=True)
        command = [*LPR_COMMAND, "infer-video", "--input", str(input_path), "--output", str(output_path), "--model", str(MODEL_PATH), "--ocr", ocr_backend.value]
        if device_input.value.strip().lower() != "auto":
            command.extend(["--device", device_input.value.strip()])
        if max_frames.value > 0:
            command.extend(["--max-frames", str(max_frames.value)])
        run(command, env=ENV)
        print(f"Annotated video: {output_path}")
        display(Video(str(output_path), embed=False, width=960))
        display(FileLink(str(output_path)))

run_button.on_click(run_video_inference)
display(widgets.VBox([video_input, video_output, widgets.HBox([ocr_backend, device_input, max_frames]), run_button, video_output_area]))

In [ ]:
# Interactive image inference and JSON output.
from IPython.display import Image as NotebookImage

image_candidates = sorted(Path("/kaggle/input").glob("**/*")) if Path("/kaggle/input").is_dir() else []
DEFAULT_IMAGE = next((str(path) for path in image_candidates if path.suffix.lower() in {".jpg", ".jpeg", ".png"}), "")
image_input = widgets.Text(value=DEFAULT_IMAGE, description="Image:", layout=widgets.Layout(width="95%"))
image_button = widgets.Button(description="Run image inference", button_style="primary", icon="search")
image_output_area = widgets.Output()

def run_image_inference(_button):
    with image_output_area:
        clear_output(wait=True)
        input_path = Path(image_input.value).expanduser()
        if not input_path.is_file():
            print(f"Image file not found: {input_path}")
            return
        display(NotebookImage(filename=str(input_path), width=960))
        command = [*LPR_COMMAND, "infer-image", "--image", str(input_path), "--model", str(MODEL_PATH), "--ocr", ocr_backend.value, "--variants", "otsu,clahe"]
        if device_input.value.strip().lower() != "auto":
            command.extend(["--device", device_input.value.strip()])
        run(command, env=ENV)

image_button.on_click(run_image_inference)
display(widgets.VBox([image_input, image_button, image_output_area]))

In [ ]:
# Optional OCR metrics. This evaluates OCR text, not detector mAP.
import csv

# Set this directly, or export LPR_OCR_CSV before running this cell:
# OCR_CSV_PATH = "/kaggle/input/<dataset-name>/ocr-ground-truth.csv"
OCR_CSV_PATH = os.getenv("LPR_OCR_CSV", "")

if not OCR_CSV_PATH and Path("/kaggle/input").is_dir():
    csv_candidates = []
    for candidate in sorted(Path("/kaggle/input").glob("**/*.csv")):
        try:
            with candidate.open(newline="", encoding="utf-8") as file:
                columns = set(csv.DictReader(file).fieldnames or [])
        except (OSError, UnicodeDecodeError):
            continue
        if {"ground_truth", "prediction"} <= columns:
            csv_candidates.append(candidate)

    if len(csv_candidates) == 1:
        OCR_CSV_PATH = str(csv_candidates[0])
        print(f"Auto-selected OCR CSV: {OCR_CSV_PATH}")
    elif csv_candidates:
        print("Multiple OCR CSV files found; set OCR_CSV_PATH explicitly:")
        for candidate in csv_candidates:
            print(f"  {candidate}")

if not OCR_CSV_PATH:
    print(
        "OCR evaluation skipped. Set OCR_CSV_PATH or LPR_OCR_CSV "
        "to a CSV containing ground_truth,prediction columns."
    )
else:
    ocr_csv_path = Path(OCR_CSV_PATH).expanduser()
    if not ocr_csv_path.is_file():
        print(f"OCR evaluation skipped; file not found: {ocr_csv_path}")
    else:
        run([*LPR_COMMAND, "evaluate-ocr", "--csv", str(ocr_csv_path)], env=ENV)

## Workflow

1. Add a Kaggle Dataset containing `best.pt`.
2. Add a Kaggle Dataset containing test images/videos.
3. Run setup cells; do not run any training cell.
4. Use the video/image widgets to inspect predictions.
5. Download the annotated video from the displayed output link.